In [1]:
# # Run this cell once, then restart the kernel before continuing.
# # MiniCPM-V 2.6 was released against Transformers 4.40.x.
%pip install -U "transformers==4.40.2" "huggingface_hub<1.0" accelerate sentencepiece safetensors opencv-python pandas pillow tqdm

print("Dependencies installed. Restart the kernel before loading the model.")

# import sys
# import subprocess

# print("Installing into:", sys.executable)

# subprocess.check_call([
#     sys.executable, "-m", "pip", "install",
#     "--no-deps",
#     "transformers==4.40.2",
#     "tokenizers==0.19.1",
#     "accelerate==0.26.1",
# ])

# subprocess.check_call([
#     sys.executable, "-m", "pip", "install",
#     "huggingface_hub<1.0",
#     "sentencepiece",
#     "safetensors",
#     "opencv-python",
#     "pandas",
#     "pillow",
#     "tqdm",
# ])

  Using cached transformers-4.40.2-py3-none-any.whl.metadata (137 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tokenizers-0.19.1-cp312-none-win_amd64.whl.metadata (6.9 kB)
Using cached transformers-4.40.2-py3-none-any.whl (9.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.19.1-cp312-none-win_amd64.whl (2.2 MB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)

   ---------------------------------------- 0/4 [opencv-python]

Note: you may need to restart the kernel to use updated packages.
Dependencies installed. Restart the kernel before loading the model.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Data\\Sequence_model\\facs-openface-tools\\.venv\\Lib\\site-packages\\cv2\\cv2.pyd'
Check the permissions.



In [2]:
pip install transformers -U

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "ipywidgets"
])

0

In [2]:
import sys
print(sys.executable)

c:\Data\Sequence_model\VLM_experiments\.venv\Scripts\python.exe


In [1]:
import importlib.util
import torch
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("flash-attn:", importlib.util.find_spec("flash_attn"))  # None is expected

ModuleNotFoundError: No module named 'transformers'

In [5]:
import ast
import gc
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, Optional

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


class MiniCPMConfig:
    model_name = "openbmb/MiniCPM-V-2_6"

    input_video = Path(
        r"C:\Data\Sequence_model\facs-openface-tools\401033_S1 - Trim.mp4"
    )
    output_video = Path("minicpm_behaviour_output.mp4")
    output_csv = Path("minicpm_behaviour_results.csv")

    # Start conservatively. Reduce to 0.3 after the 60-second test works.
    analysis_interval_seconds = 1.0
    test_duration_seconds: Optional[float] = 60.0

    # Pre-resizing reduces visual tokens and inference time.
    max_image_width = 768
    max_new_tokens = 220


if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the RTX 5090 Python environment.")

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
supported_arches = torch.cuda.get_arch_list()
free_vram, total_vram = torch.cuda.mem_get_info()

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", gpu_name)
print("Compute capability:", gpu_capability)
print(f"VRAM: {free_vram / 1024**3:.1f} GB free / {total_vram / 1024**3:.1f} GB total")
print("PyTorch CUDA architectures:", supported_arches)

if gpu_capability >= (12, 0) and "sm_120" not in supported_arches:
    raise RuntimeError(
        "This PyTorch build does not contain RTX 5090 (sm_120) kernels. "
        "Install a current CUDA 13.0 build as shown in the markdown cell, "
        "then restart the kernel."
    )

if not MiniCPMConfig.input_video.exists():
    raise FileNotFoundError(f"Input video not found: {MiniCPMConfig.input_video}")

PyTorch: 2.13.0+cu130
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5090
Compute capability: (12, 0)
VRAM: 30.2 GB free / 31.8 GB total
PyTorch CUDA architectures: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [6]:
BEHAVIOUR_PROMPT = r"""
Analyse only directly visible behaviour in this video frame.
Focus on the most prominent person. Do not infer emotion, intention,
personality, diagnosis, or any fact that is not visually observable.

Return exactly one JSON object with this schema:
{
  "face_visible": true,
  "expression": "neutral",
  "gaze": "looking away",
  "head_pose": "turned left",
  "eyes": "open",
  "mouth": "closed",
  "engagement": "unclear",
  "behaviour": "brief factual description",
  "confidence": 0.75
}

Allowed expression values: smiling, slight smile, neutral, frowning,
tense, surprised, unclear.
Allowed gaze values: toward camera, left, right, up, down, looking away,
eyes closed, unclear.
Allowed head_pose values: frontal, turned left, turned right, tilted left,
tilted right, lowered, raised, unclear.
Allowed engagement values: attentive, withdrawn, active, unclear.
Return JSON only, without Markdown.
""".strip()


def default_result() -> Dict[str, Any]:
    return {
        "face_visible": False,
        "expression": "unclear",
        "gaze": "unclear",
        "head_pose": "unclear",
        "eyes": "unclear",
        "mouth": "unclear",
        "engagement": "unclear",
        "behaviour": "unclear",
        "confidence": 0.0,
    }


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")
    if first_brace < 0 or last_brace <= first_brace:
        return None

    candidate = cleaned[first_brace:last_brace + 1]
    try:
        parsed = json.loads(candidate)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    try:
        parsed = ast.literal_eval(candidate)
        return parsed if isinstance(parsed, dict) else None
    except (ValueError, SyntaxError):
        return None


def normalise_result(parsed: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    result = default_result()
    if parsed:
        for key in result:
            if key in parsed:
                result[key] = parsed[key]

    visible = result["face_visible"]
    if isinstance(visible, str):
        visible = visible.strip().lower() == "true"
    result["face_visible"] = bool(visible)

    try:
        result["confidence"] = float(np.clip(float(result["confidence"]), 0, 1))
    except (TypeError, ValueError):
        result["confidence"] = 0.0

    for key in ("expression", "gaze", "head_pose", "eyes",
                "mouth", "engagement", "behaviour"):
        result[key] = str(result[key]).strip()
    return result


def resize_for_model(frame: np.ndarray, max_width: int) -> np.ndarray:
    height, width = frame.shape[:2]
    if width <= max_width:
        return frame
    scale = max_width / width
    return cv2.resize(
        frame,
        (max_width, max(1, round(height * scale))),
        interpolation=cv2.INTER_AREA,
    )

In [7]:
from huggingface_hub import HfApi, hf_hub_download, login

try:
    from huggingface_hub import get_token
except ImportError:
    from huggingface_hub import HfFolder

    def get_token():
        return HfFolder.get_token()

try:
    from huggingface_hub.errors import GatedRepoError
except ImportError:
    from huggingface_hub.utils import GatedRepoError

MODEL_PAGE = "https://huggingface.co/openbmb/MiniCPM-V-2_6"

# Plain login() works with both older and newer huggingface_hub releases.
# It is called only when this Python environment has no saved token.
if get_token() is None:
    login()
account = HfApi().whoami()
print("Hugging Face account:", account.get("name", account))

try:
    config_path = hf_hub_download(
        repo_id=MiniCPMConfig.model_name,
        filename="config.json",
        token=True,
    )
    print("MiniCPM-V access confirmed:", config_path)
except GatedRepoError as exc:
    raise PermissionError(
        "This Hugging Face account does not yet have access to MiniCPM-V 2.6. "
        f"Open {MODEL_PAGE}, submit/accept the access request, and rerun this cell."
    ) from exc

Hugging Face account: Wasifjafri
MiniCPM-V access confirmed: C:\Users\compu\.cache\huggingface\hub\models--openbmb--MiniCPM-V-2_6\snapshots\6c04d9e3022bcff6e6738dfb1fc19a5cfd2a855f\config.json


In [8]:
import torch
import transformers.dynamic_module_utils as dmu
from transformers import AutoModel, AutoTokenizer

# MiniCPM conditionally imports flash_attn, but Transformers 4.40.2's
# static dependency scanner incorrectly treats it as mandatory.
if not hasattr(dmu, "_original_get_imports_minicpm"):
    dmu._original_get_imports_minicpm = dmu.get_imports

def _get_imports_without_optional_flash_attn(filename):
    imports = dmu._original_get_imports_minicpm(filename)
    return [
        package for package in imports
        if package.split(".")[0] != "flash_attn"
    ]

dmu.get_imports = _get_imports_without_optional_flash_attn

model_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    MiniCPMConfig.model_name,
    trust_remote_code=True,
    token=True,
)

model = AutoModel.from_pretrained(
    MiniCPMConfig.model_name,
    trust_remote_code=True,
    token=True,
    attn_implementation="sdpa",
    torch_dtype=model_dtype,
    low_cpu_mem_usage=True,
).eval().cuda()

print("Loaded on:", next(model.parameters()).device)
print("Attention:", model.config._attn_implementation)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/789 [00:00<?, ?it/s]

AttributeError: 'MiniCPMV' object has no attribute 'all_tied_weights_keys'

In [ ]:
# The official MiniCPM-V 2.6 example supports SDPA or FlashAttention 2.
# SDPA is used here because it works on Windows without compiling flash-attn.
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Loading:", MiniCPMConfig.model_name)
print("Model dtype:", model_dtype)

tokenizer = AutoTokenizer.from_pretrained(
    MiniCPMConfig.model_name,
    trust_remote_code=True,
    token=True,
)

model = AutoModel.from_pretrained(
    MiniCPMConfig.model_name,
    trust_remote_code=True,
    token=True,
    attn_implementation="sdpa",
    torch_dtype=model_dtype,
    low_cpu_mem_usage=True,
).eval().cuda()

print("MiniCPM-V 2.6 loaded on", next(model.parameters()).device)

In [ ]:
def analyse_frame_with_minicpm(frame_bgr: np.ndarray) -> Dict[str, Any]:
    if frame_bgr is None or frame_bgr.size == 0:
        return default_result()

    prepared = resize_for_model(frame_bgr, MiniCPMConfig.max_image_width)
    image = Image.fromarray(cv2.cvtColor(prepared, cv2.COLOR_BGR2RGB))
    messages = [{"role": "user", "content": [image, BEHAVIOUR_PROMPT]}]

    try:
        with torch.inference_mode():
            response = model.chat(
                image=None,
                msgs=messages,
                tokenizer=tokenizer,
                sampling=False,
                max_new_tokens=MiniCPMConfig.max_new_tokens,
            )

        parsed = extract_json_object(str(response))
        if parsed is None:
            print("Could not parse MiniCPM response:", response)
        return normalise_result(parsed)

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("CUDA out of memory. Reduce max_image_width or increase the interval.")
        return default_result()
    except Exception as exc:
        print(f"MiniCPM frame-analysis error: {type(exc).__name__}: {exc}")
        return default_result()


# Single-frame smoke test before processing the whole video.
test_capture = cv2.VideoCapture(str(MiniCPMConfig.input_video))
test_ok, test_frame = test_capture.read()
test_capture.release()
if not test_ok:
    raise RuntimeError("Could not read the first frame from the input video.")

smoke_test_result = analyse_frame_with_minicpm(test_frame)
print(json.dumps(smoke_test_result, indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path
import subprocess
import cv2

source = MiniCPMConfig.input_video.resolve()
converted = source.with_name(f"{source.stem}_opencv_h264.mp4")

command = [
    "ffmpeg", "-y",
    "-i", str(source),
    "-map", "0:v:0",
    "-map", "0:a:0?",
    "-c:v", "libx264",
    "-crf", "18",
    "-preset", "medium",
    "-pix_fmt", "yuv420p",
    "-vsync", "cfr",
    "-c:a", "aac",
    "-b:a", "192k",
    "-ar", "48000",
    "-movflags", "+faststart",
    str(converted),
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

if result.returncode != 0:
    print(result.stderr[-5000:])
    raise RuntimeError("FFmpeg conversion failed; see the output above.")

print("Converted:", converted)
print("Size:", converted.stat().st_size, "bytes")

# Verify OpenCV decoding before continuing.
capture = cv2.VideoCapture(str(converted), cv2.CAP_FFMPEG)
opened = capture.isOpened()
test_ok, test_frame = capture.read()
capture.release()

print("Opened:", opened)
print("First frame decoded:", test_ok)
print("Frame shape:", None if test_frame is None else test_frame.shape)

if not opened or not test_ok or test_frame is None:
    raise RuntimeError("The converted video still cannot be decoded by OpenCV.")

MiniCPMConfig.input_video = converted

smoke_test_result = analyse_frame_with_minicpm(test_frame)
print(json.dumps(smoke_test_result, indent=2, ensure_ascii=False))